### IMPORTING LIBRARY

In [ ]:
import os
import random
import datetime
import json
import umap
import pandas                   as pd
import numpy                    as np
import plotly.express           as px
import matplotlib.pyplot        as plt

from tqdm                       import tqdm
from preProcessingData          import PreProcessData
from utility.matpilotDraw       import MatPlot
from prettytable                import PrettyTable
from model.vaeConv1DCustom      import vaeConv1DCustom 
from vaeTrainer                 import appVAE
from mpl_toolkits.mplot3d       import Axes3D
from sklearn.cluster            import KMeans


matPlot    = MatPlot() 
preData    = PreProcessData()

### DATA LOADING

In [ ]:
rawData = preData.dataLoading()

In [ ]:
preData.save2H5Format(rawData)

In [ ]:
data = preData.loadDataFromH5Format()
print(data.info())

### DATA CLEANING

#### Remove redundant mode of failure (option)

In [ ]:
preData.rmRedundantMOF()

#### Manipulated and efficiency Variables

In [ ]:
preData.splitInputandEfficiencyVar()

##### Draw variables

In [ ]:
# ============================
# Manipulated variables
# ----------------------------
for idxFeature in preData.manipulatedCr1:
    drawedData = preData.crusher1[idxFeature].to_numpy().astype(float)
    matPlot.draw1D(yCoor=drawedData,
                   title=idxFeature)

In [ ]:
# ============================
# Efficiency variables
# ----------------------------
for idxFeature in preData.efficiencyCr1:
    drawedData = preData.crusher1[idxFeature].to_numpy().astype(float)
    matPlot.draw1D(yCoor=drawedData,
                   title=idxFeature)

#### Sumarry MOF

In [ ]:
preData.sumarryMOF()

#### Data summarization

In [ ]:
preData.sumarryDataFromFile()

## BUIDING MODEL

In [ ]:
norData = preData.normalizeData(rawData[preData.crusher1Value])    # Normalize the data

In [ ]:
# ============================
# Manipulated variables
# ----------------------------
for idxFeature in preData.manipulatedCr1:
    drawedData = norData[idxFeature].to_numpy().astype(float)
    matPlot.draw1D(yCoor=drawedData,
                   title=idxFeature)

In [ ]:
# ============================
# Efficiency variables
# ----------------------------
for idxFeature in preData.efficiencyCr1:
    drawedData = norData[idxFeature].to_numpy().astype(float)
    matPlot.draw1D(yCoor=drawedData,
                   title=idxFeature)

### =================================
### Training


In [ ]:
healthyStateData        = preData.getHealthyState(norData)

In [ ]:
# curData         = rawData[preData.crusher1Value]
# greaterThan500  = curData["corriente(A)"]== 0
# indices         = greaterThan500[greaterThan500].index.tolist()
# print(indices)
curData         = norData

healthyExcellPos = {"startingRow":  [18317, 28175,  43739,  64526,  87163,  107316, 115185, 133833, 225442, 280493, 310291], #277099 instead of 280493, but the current values is 0
                    "endingRow":    [27068, 42727,  63051,  86359,  98506,  115132, 132599, 157742, 249835, 283708, 321436],
                    "percent":      [0.5,   0.5,    0.5,    0.5,    0.5,    0.5,    0.5,    0.5,    0.3,    0.5,    0.7]}   # should subtract 2

healthyData =  pd.DataFrame(columns=preData.crusher1Value)
for idx in range(len(healthyExcellPos["startingRow"])):
    lenOfHealthyRow     = int((healthyExcellPos["endingRow"][idx] - healthyExcellPos["startingRow"][idx]) * healthyExcellPos["percent"][idx])

    startingPos         = healthyExcellPos["startingRow"][idx] -2
    endingPos           = healthyExcellPos["startingRow"][idx] - 2 + lenOfHealthyRow
    htEachPeriod        = curData[startingPos:endingPos]         
    healthyData         = pd.concat([healthyData,htEachPeriod],ignore_index=True)

In [ ]:
# ============================
# Manipulated variables
# ----------------------------
for idxFeature in preData.manipulatedCr1:
    drawedData = healthyStateData[idxFeature].to_numpy().astype(float)
    matPlot.draw1D(yCoor=drawedData,
                   title=idxFeature)

In [ ]:
inputX                  = healthyStateData[preData.manipulatedCr1]
outputY                 = healthyStateData[preData.efficiencyCr1]

trainXY,valXY           = preData.splitTrainVal(inputX,outputY)                     # Split input and efficiency variables into train and validation dataset
tensorXY                = preData.splitAndPadSequences(trainXY)                # Split them into sequences

print(len(trainXY[0]))
print(tensorXY[1].shape)

In [ ]:
seqLength       = 30
overlap         = 29

sequences       = []
step_size       = seqLength - overlap
totalLength     = len(trainX)

for start in range(0, totalLength-overlap, step_size):
    end = start + seqLength
    # ----------------------
    # If the sequence would 
    # go beyond the data length, 
    # pad it
    # ----------------------
    if end > totalLength:                
        padding  = end - totalLength
        # sequence = torch.nn.functional.pad(dataX[:, start:], (0, padding), 'constant', 0)
    
    else:
        sequence = trainX[start:end].to_numpy()           # Get sentense
        
    sequences.append(sequence)


tensorX =   np.stack(sequences, axis=0)    
print(tensorX.shape)


#### Model

In [ ]:
with open('model/modelConfig.json', 'r') as configFile:
    config = json.load(configFile)

vaeConfig = config["vae"]
 
inputX                  = healthyStateData[preData.manipulatedCr1]
outputY                 = healthyStateData[preData.efficiencyCr1]

trainXY,valXY           = preData.splitTrainVal(inputX,outputY)                     # Split input and efficiency variables into train and validation dataset

seqTrainXY              = preData.splitAndPadSequences(trainXY)                     # Split them into sequences
seqValXY                = preData.splitAndPadSequences(valXY)                       # Split them into sequences

model                   = vaeConv1DCustom(vaeConfig)

print("The size of train data:  ",seqTrainXY[0].shape ,"(input)    ", seqTrainXY[1].shape,"(output)")
print("The size of val data:    ",seqValXY[0].shape ,"(input)    ", seqValXY[1].shape,"(output)")

model.summary()

### GET CONSTRUCTION ERROR AND LATENT SPACE

In [ ]:
inputX                  = norData[preData.manipulatedCr1]
outputY                 = norData[preData.efficiencyCr1]
seqData                 = preData.splitAndPadSequences([inputX,outputY])                       # Split them into sequences

lossResults,latentResults   = appVAE.predict(seqData,vaeConfig)

### D. UMAP for entire dataset

In [ ]:
manipulatedCr1  = preData.crusher1[preData.manipulatedCr1]
maWTimeCr1      = manipulatedCr1[:]
print("The input variables shape",maWTimeCr1.shape)
umapReduction = umap.UMAP(  n_components    = 3,
                            n_neighbors     = 15,
                            min_dist        = 0.1)
maEmbedding     = umapReduction.fit_transform(maWTimeCr1)

In [ ]:
kmeans = KMeans(n_clusters=4)
kmeans.fit(maEmbedding)

labels = kmeans.labels_
centers = kmeans.cluster_centers_

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

# Color map for clusters
colors = ['r', 'g', 'b', 'y']

# Plot each data point
for i in range(len(maEmbedding)):
    ax.scatter(maEmbedding[i, 0], maEmbedding[i, 1], maEmbedding[i, 2], c=colors[labels[i]], marker='o')

# Plot the centroids
for i in range(4):
    ax.scatter(centers[i, 0], centers[i, 1], centers[i, 2], c=colors[i], marker='x', s=100, linewidths=3)

ax.set_title('Mode of condition')
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.set_zlabel('Feature 3')
plt.show()

In [ ]:
efficiencyCr1   = preData.crusher1[preData.efficiencyCr1]
effWTimeCr1     = efficiencyCr1[:]
print("The efficieny variables shape",effWTimeCr1.shape)

umapReduction1 = umap.UMAP(  n_components=3,
                            n_neighbors=15,
                            min_dist=0.1)

effEmbedding      = umapReduction1.fit_transform(effWTimeCr1)

In [ ]:
kmeans1 = KMeans(n_clusters=4)
kmeans1.fit(effEmbedding)

labels = kmeans1.labels_
centers = kmeans1.cluster_centers_

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

# Color map for clusters
colors = ['r', 'g', 'b', 'y']

# Plot each data point
for i in range(len(effEmbedding)):
    ax.scatter(effEmbedding[i, 0], effEmbedding[i, 1], effEmbedding[i, 2], c=colors[labels[i]], marker='o')

# Plot the centroids
for i in range(4):
    ax.scatter(centers[i, 0], centers[i, 1], centers[i, 2], c=colors[i], marker='x', s=100, linewidths=3)

ax.set_title('Mode of condition')
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.set_zlabel('Feature 3')
plt.show()

#### c.1 Draw UMAP

In [ ]:
matPlot.draw3D(yCoor1 = maEmbedding[:,0],
               yCoor2 = maEmbedding[:,1],
               yCoor3 = maEmbedding[:,2],
               title  = "UMAP for Manipulated variables")

matPlot.draw3D(yCoor1 = effEmbedding[:,0],
               yCoor2 = effEmbedding[:,1],
               yCoor3 = effEmbedding[:,2],
               title  = "UMAP for efficient variables")

In [ ]:
maEmbedding_df  = pd.DataFrame(maEmbedding, columns=['UMAP_1', 'UMAP_2', 'UMAP_3'])
effEmbedding_df = pd.DataFrame(effEmbedding, columns=['UMAP_1', 'UMAP_2', 'UMAP_3'])
print(maEmbedding_df)
print(effEmbedding_df)
fig = px.scatter_3d(
    maEmbedding_df,
    x='UMAP_1',
    y='UMAP_2',
    z='UMAP_3',
    title='3D UMAP Embedding with Plotly'
)

fig.update_traces(marker=dict(size=3))
fig.show()

fig = px.scatter_3d(
    effEmbedding_df,
    x='UMAP_1',
    y='UMAP_2',
    z='UMAP_3',
    title='3D UMAP Embedding with Plotly'
)

fig.update_traces(marker=dict(size=3))
fig.show()